# 1 Pandas高级数据处理

In [2]:
import numpy as np
import pandas as pd

## 1.1 级联

汇总全年销售数据

In [3]:
df1 = pd.DataFrame(data=[[1,2,3]], columns=list('ABC'))
df1

,A,B,C
0,1,2,3


In [4]:
df2 = pd.DataFrame(data=[[2,3,4]], columns=list('ABC'))
df2

,A,B,C
0,2,3,4


In [5]:
# 级联语法的核心就是索引对齐
# 级联的应用场景：不同期，但是结构相同的数据汇总
# object对象
# axis=0列索引对齐，axis=1行索引对齐
res1 = pd.concat([df1, df2], axis=0)

In [6]:
res2 = pd.concat([df1, df2], axis=1)

In [7]:
res1.loc[0]

,A,B,C
0,1,2,3
0,2,3,4


In [8]:
res2.loc[:, 'A']

,A,A
0,1,2


In [9]:
# 校验级联之后是否有重复索引
# pd.concat((df1, df2), verify_integrity=True)

In [10]:
# pd.concat((df1, df2), axis=1, verify_integrity=True)

In [11]:
# 通过多层级索引来处理重复索引的问题
pd.concat((df1, df2), axis=1, keys=['上半年', '下半年'], names=['周期', '产品'])

周期 上半年       下半年      
产品   A  B  C   A  B  C
0    1  2  3   2  3  4

In [12]:
# 通过忽略索引的方式来处理重复索引的问题
pd.concat((df1, df2), ignore_index=True)

,A,B,C
0,1,2,3
1,2,3,4


In [13]:
df3 = pd.DataFrame(data=[[1,2,3,4]], columns=list('CDAB'))
df3

,C,D,A,B
0,1,2,3,4


In [14]:
df2

,A,B,C
0,2,3,4


In [15]:
pd.concat((df3, df2), sort=False)

,C,D,A,B
0,1,2.0,3,4
0,4,NaN,2,3


In [16]:
df4=pd.DataFrame(data=np.random.randint(0,100,size=(3,4)),columns=list('ABCD'))
df5=pd.DataFrame(data=np.random.randint(-100,0,size=(4,3)),columns=list('BDE'))

In [17]:
display(df4, df5)

,A,B,C,D
0,21,78,45,37
1,96,32,69,59
2,41,88,60,82


,B,D,E
0,-35,-87,-15
1,-53,-23,-95
2,-49,-56,-21
3,-35,-8,-94


In [18]:
# outer 保留级联方向的所有【标签】并集
# inner 保留级联方向的共有【标签】交集
pd.concat((df4, df5, df1), sort=True, join='outer')

,A,B,C,D,E
0,21.0,78,45.0,37.0,NaN
1,96.0,32,69.0,59.0,NaN
2,41.0,88,60.0,82.0,NaN
0,NaN,-35,NaN,-87.0,-15.0
1,NaN,-53,NaN,-23.0,-95.0
2,NaN,-49,NaN,-56.0,-21.0
3,NaN,-35,NaN,-8.0,-94.0
0,1.0,2,3.0,NaN,NaN


## 1.2 合并

合并就是根据两张表的公共信息，把两张表的数据汇总的方法。

合并以列的内容为参考标准，不存在行合并，都是列合并合并的列通常是离散型数据。

可以是数值型，也可以是类别型数据合并的列之间存在一对一、一对多、多对多关系，否则合并结果为空

### 1.2.1 计算上半年订单总额GMV

参数：left_onright_on

In [19]:
first_half_year=pd.read_excel('合并表格案例.xlsx',sheet_name=0)
second_half_year=pd.read_excel('合并表格案例.xlsx',sheet_name=1)
display(first_half_year.head(), second_half_year.head())

,用户ID,商品ID,订单ID,购买数量
0,lucy,10001,1009,1
1,jack,10002,1002,2
2,lucy,10003,1007,1
3,alex,10004,1010,1
4,mery,10005,1008,1


,用户ID,商品ID,订单ID,购买数量
0,tom,10006,2001,1
1,oldshang,10003,2002,1
2,佩奇,10004,2003,1
3,小明,10004,2004,2
4,小红,10001,2005,1


In [20]:
user_table=pd.read_excel('合并表格案例.xlsx',sheet_name=2)
user_table

,用户ID,地区,VIP等级,手机号
0,lucy,北京,3,13054344433
1,智哥,深圳,3,13046798795
2,mery,北京,2,17877659878
3,jack,上海,4,18635482221
4,alex,北京,1,17601002323
5,tom,深圳,2,18910538799
6,oldshang,北京,3,17699887678
7,佩奇,上海,2,15600140101
8,小明,上海,2,18789897788
9,小红,北京,3,17625745653


In [21]:
product_table = pd.read_excel('合并表格案例.xlsx', sheet_name=3)
product_table.head()

,商品ID,商品类别,商品品牌,商品单价
0,10001,笔记本,华为,8000
1,10002,笔记本,小米,7600
2,10003,鼠标,华为,300
3,10004,鼠标,apple,600
4,10005,键盘,apple,1000


In [22]:
return_table = pd.read_excel('合并表格案例.xlsx', sheet_name=4)
return_table.head()

,订单_id,退货状态
0,1003,退货中
1,1004,退货中
2,1005,退货完成
3,1011,退货完成
4,1014,退货中


In [23]:
#两张表合并时，默认是根据所有的相同字段名称的列来进行合并
res1 = pd.merge(left = first_half_year, right = product_table)
res1['订单总额'] = res1['购买数量'] * res1['商品单价']
res1

,用户ID,商品ID,订单ID,购买数量,商品类别,商品品牌,商品单价,订单总额
0,lucy,10001,1009,1,笔记本,华为,8000,8000
1,jack,10002,1002,2,笔记本,小米,7600,15200
2,lucy,10003,1007,1,鼠标,华为,300,300
3,alex,10004,1010,1,鼠标,apple,600,600
4,mery,10005,1008,1,键盘,apple,1000,1000
5,jack,10006,1003,1,键盘,小米,200,200
6,佩奇,10002,1013,1,笔记本,小米,7600,7600
7,alex,10001,1001,3,笔记本,华为,8000,24000
8,智哥,10004,1004,1,鼠标,apple,600,600
9,tom,10001,1011,1,笔记本,华为,8000,8000


### 1.2.2 获取上半年用户地区，查看各地区订单数量

参数：how

In [24]:
user_table.head()

,用户ID,地区,VIP等级,手机号
0,lucy,北京,3,13054344433
1,智哥,深圳,3,13046798795
2,mery,北京,2,17877659878
3,jack,上海,4,18635482221
4,alex,北京,1,17601002323


In [25]:
first_half_year

,用户ID,商品ID,订单ID,购买数量
0,lucy,10001,1009,1
1,jack,10002,1002,2
2,lucy,10003,1007,1
3,alex,10004,1010,1
4,mery,10005,1008,1
5,jack,10006,1003,1
6,佩奇,10002,1013,1
7,alex,10001,1001,3
8,智哥,10004,1004,1
9,tom,10001,1011,1


In [26]:
pd.merge(left=first_half_year, right=user_table)

,用户ID,商品ID,订单ID,购买数量,地区,VIP等级,手机号
0,lucy,10001,1009,1,北京,3,13054344433
1,jack,10002,1002,2,上海,4,18635482221
2,lucy,10003,1007,1,北京,3,13054344433
3,alex,10004,1010,1,北京,1,17601002323
4,mery,10005,1008,1,北京,2,17877659878
5,jack,10006,1003,1,上海,4,18635482221
6,佩奇,10002,1013,1,上海,2,15600140101
7,alex,10001,1001,3,北京,1,17601002323
8,智哥,10004,1004,1,深圳,3,13046798795
9,tom,10001,1011,1,深圳,2,18910538799


In [28]:
# 'left', 'right', 'outer', 'inner' 基于列的内容
# inner 只保留合并后列内的交集
# outer 保留合并列内的并集
pd.merge(left=first_half_year, right=user_table, how='inner')['地区'].value_counts()

地区
北京    6
上海    3
深圳    2
Name: count, dtype: int64

### 1.2.3 找出上半年和下半年购买过相同商品的用户

参数解释：on，suffixes

一个用户，上半年和下半年都购买了同一个商品

In [ ]:
first_half_year

,用户ID,商品ID,订单ID,购买数量
0,tom,10006,2001,1
1,oldshang,10003,2002,1
2,佩奇,10004,2003,1
3,小明,10004,2004,2
4,小红,10001,2005,1
5,大王,10006,2006,1
6,lucy,10003,2007,1
7,佩奇,10002,2008,1
8,小明,10002,2009,2
9,mery,10005,2010,1


In [30]:
second_half_year

,用户ID,商品ID,订单ID,购买数量
0,tom,10006,2001,1
1,oldshang,10003,2002,1
2,佩奇,10004,2003,1
3,小明,10004,2004,2
4,小红,10001,2005,1
5,大王,10006,2006,1
6,lucy,10003,2007,1
7,佩奇,10002,2008,1
8,小明,10002,2009,2
9,mery,10005,2010,1


In [33]:
pd.merge(left=first_half_year, right=second_half_year,
on=["用户ID", "商品ID"],
suffixes=["_下半年","_上半年"])

,用户ID,商品ID,订单ID_下半年,购买数量_下半年,订单ID_上半年,购买数量_上半年
0,lucy,10003,1007,1,2007,1
1,mery,10005,1008,1,2010,1
2,佩奇,10002,1013,1,2008,1
3,智哥,10004,1004,1,2011,3


### 1.2.4	查看上半年退货商品总额

参数：left_on，right_on，left_index，right_index

In [34]:
return_table

,订单_id,退货状态
0,1003,退货中
1,1004,退货中
2,1005,退货完成
3,1011,退货完成
4,1014,退货中
5,1007,退货完成


In [35]:
first_half_year

,用户ID,商品ID,订单ID,购买数量
0,lucy,10001,1009,1
1,jack,10002,1002,2
2,lucy,10003,1007,1
3,alex,10004,1010,1
4,mery,10005,1008,1
5,jack,10006,1003,1
6,佩奇,10002,1013,1
7,alex,10001,1001,3
8,智哥,10004,1004,1
9,tom,10001,1011,1


In [42]:
res2 = pd.merge(left = first_half_year, right= return_table, left_on='订单ID' , right_on = '订单_id')

In [43]:
res2

,用户ID,商品ID,订单ID,购买数量,订单_id,退货状态
0,lucy,10003,1007,1,1007,退货完成
1,jack,10006,1003,1,1003,退货中
2,智哥,10004,1004,1,1004,退货中
3,tom,10001,1011,1,1011,退货完成


In [44]:
product_table

,商品ID,商品类别,商品品牌,商品单价
0,10001,笔记本,华为,8000
1,10002,笔记本,小米,7600
2,10003,鼠标,华为,300
3,10004,鼠标,apple,600
4,10005,键盘,apple,1000
5,10006,键盘,小米,200


In [48]:
res3 = pd.merge(left=res2, right=product_table, how='inner')
(res3['购买数量'] * res3['商品单价']).sum()

np.int64(9100)

### 1.2.5 汇总全年订单数据

In [51]:
total = pd.concat((first_half_year, second_half_year), ignore_index=True)

### 1.2.6 计算全年客单价

每一单成交金额均值

In [53]:
res4 = pd.merge(left=total, right=product_table)
(res4['购买数量'] * res4['商品单价']).mean()

np.float64(5178.260869565217)

## 1.3 分组

分组必聚合

In [58]:
df = pd.read_excel('分组表格案例.xlsx', index_col=0)

In [63]:
df.head()

,菜品,颜色,价格,数量
Unnamed: 0,,,,
0,白菜,绿,64,88
1,白菜,红,24,98
2,冬瓜,红,0,43
3,辣椒,红,59,51
4,西红柿,白,29,29


groupby()

In [71]:
# DataFrameGroupBy 对象可以直接响应pandas的聚合函数
# 注意：聚合方法只能对数值类型有效
df.groupby(by=['菜品'])['数量'].sum()

菜品
冬瓜     159
白菜     316
茄子     315
西红柿     93
辣椒     309
青椒     168
Name: 数量, dtype: int64

In [72]:
df.groupby(by=['菜品'])['价格'].mean()

菜品
冬瓜     51.000000
白菜     33.666667
茄子     49.400000
西红柿    33.000000
辣椒     40.285714
青椒     61.666667
Name: 价格, dtype: float64

groups

In [73]:
df.groupby(by=['菜品']).groups

{'冬瓜': [2, 6, 7, 8, 19, 23], '白菜': [0, 1, 5, 13, 18, 24], '茄子': [14, 16, 20, 22, 29], '西红柿': [4, 10, 26], '辣椒': [3, 9, 11, 12, 15, 25, 28], '青椒': [17, 21, 27]}

In [74]:
df.loc[[0, 1, 5, 13, 17, 25, 26, 27]]

,菜品,颜色,价格,数量
Unnamed: 0,,,,
0,白菜,绿,64,88
1,白菜,红,24,98
5,白菜,白,5,37
13,白菜,红,66,11
17,青椒,绿,85,59
25,辣椒,白,15,98
26,西红柿,绿,58,15
27,青椒,白,90,84


多分组

In [ ]:
df.groupby(['菜品', '颜色'])['价格'].mean().unstack()

颜色,白,紫,红,绿
菜品,,,,
冬瓜,65.5,NaN,31.333333,81.0
白菜,5.0,22.0,45.000000,42.5
茄子,NaN,45.5,47.000000,54.5
西红柿,29.0,NaN,12.000000,58.0
辣椒,15.0,NaN,44.500000,NaN
青椒,50.0,NaN,NaN,85.0


In [81]:
df.groupby(['菜品', '颜色'])['数量'].sum()

菜品   颜色
冬瓜   白      32
     红      97
     绿      30
白菜   白      37
     紫      50
     红     109
     绿     120
茄子   紫     102
     红      15
     绿     198
西红柿  白      29
     红      49
     绿      15
辣椒   白      98
     红     211
青椒   白     109
     绿      59
Name: 数量, dtype: int64

定制多重聚合指标

In [77]:
gpobj = df.groupby('菜品')

# agg接收一个字典对象
#字典对象里：键是要聚合的列名称，值是一个函数名字（地址），是一个聚合函数
gpobj.agg({
    '价格':np.mean,
    '数量':np.sum
})

C:\Users\songlk2\AppData\Local\Temp\ipykernel_36012\3934893420.py:5: FutureWarning: The provided callable <function mean at 0x000001793CF4DDA0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  gpobj.agg({
C:\Users\songlk2\AppData\Local\Temp\ipykernel_36012\3934893420.py:5: FutureWarning: The provided callable <function sum at 0x000001793CF4C9A0> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  gpobj.agg({


,价格,数量
菜品,,
冬瓜,51.000000,159
白菜,33.666667,316
茄子,49.400000,315
西红柿,33.000000,93
辣椒,40.285714,309
青椒,61.666667,168


高级聚合

In [82]:
# df.groupby('菜品')['价格'].mean()
df.groupby('菜品')['价格'].apply(np.mean)

菜品
冬瓜     51.000000
白菜     33.666667
茄子     49.400000
西红柿    33.000000
辣椒     40.285714
青椒     61.666667
Name: 价格, dtype: float64

In [ ]:
def my_mean(x):
    # 对于分组对象而言，apply传递的函数，接收到的参数x是一组数据
    return x.mean()

In [87]:
df.groupby('菜品')['价格'].apply(my_mean)

菜品
冬瓜     51.000000
白菜     33.666667
茄子     49.400000
西红柿    33.000000
辣椒     40.285714
青椒     61.666667
Name: 价格, dtype: float64

## 1.4 交叉表

交叉表统计的数量 count

In [89]:
df.groupby(['菜品', '颜色'])['数量'].count().unstack()

颜色,白,紫,红,绿
菜品,,,,
冬瓜,2.0,NaN,3.0,1.0
白菜,1.0,1.0,2.0,2.0
茄子,NaN,2.0,1.0,2.0
西红柿,1.0,NaN,1.0,1.0
辣椒,1.0,NaN,6.0,NaN
青椒,2.0,NaN,NaN,1.0


In [93]:
# index 是一个序列，而不是一个列标签
# columns
pd.crosstab(index=df['菜品'], columns=df['颜色'])

颜色,白,紫,红,绿
菜品,,,,
冬瓜,2,0,3,1
白菜,1,1,2,2
茄子,0,2,1,2
西红柿,1,0,1,1
辣椒,1,0,6,0
青椒,2,0,0,1


## 1.5 透视表

In [94]:
# data 数据源 就是要进行统计的DataFrame对象
# index\columns 透视表的行列是从数据源的哪列提取的
# values是要统计的数据源中的字段
# aggfunc是聚合方法，传的是函数名字
pd.pivot_table(data=df, index='菜品', columns='颜色', values=['价格'], aggfunc=np.mean)

C:\Users\songlk2\AppData\Local\Temp\ipykernel_36012\801006404.py:5: FutureWarning: The provided callable <function mean at 0x000001793CF4DDA0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  pd.pivot_table(data=df, index='菜品', columns='颜色', values=['价格'], aggfunc=np.mean)


价格                       
颜色      白     紫          红     绿
菜品                              
冬瓜   65.5   NaN  31.333333  81.0
白菜    5.0  22.0  45.000000  42.5
茄子    NaN  45.5  47.000000  54.5
西红柿  29.0   NaN  12.000000  58.0
辣椒   15.0   NaN  44.500000   NaN
青椒   50.0   NaN        NaN  85.0